In [1]:
import requests

API_KEY = "csWNCWjgntYlmN2wcyNi9HZOX4MImXbBNbLtoQDc"

url = f"https://api.nasa.gov/neo/rest/v1/feed?start_date=2024-01-01&end_date=2024-01-07&api_key={API_KEY}"
response = requests.get(url)

In [2]:
response

<Response [200]>

In [3]:
data = response.json()

In [4]:
data['links']['next']

'http://api.nasa.gov/neo/rest/v1/feed?start_date=2024-01-07&end_date=2024-01-13&detailed=false&api_key=csWNCWjgntYlmN2wcyNi9HZOX4MImXbBNbLtoQDc'

In [9]:
data.keys()

dict_keys(['links', 'element_count', 'near_earth_objects'])

In [11]:
details = data['near_earth_objects']

In [13]:
len(details)

7

In [ ]:
import requests
from datetime import datetime

all_asteroids = []
row_limit = 10000

# Set your initial API call URL
API_KEY = "csWNCWjgntYlmN2wcyNi9HZOX4MImXbBNbLtoQDc"  # Replace with your actual NASA API key
start_url = f"https://api.nasa.gov/neo/rest/v1/feed?start_date=2024-01-01&end_date=2024-01-07&api_key={API_KEY}"

while start_url and len(all_asteroids) < row_limit:
    response = requests.get(start_url)
    data = response.json()
    details = data['near_earth_objects']

    for date, info in details.items():
        for i in info:
            approach_data = i['close_approach_data'][0]

            asteroid_data = {
                'id': i['id'],
                'name': i['name'],
                'absolute_magnitude_h': i['absolute_magnitude_h'],
                'est_diameter_min_km': i['estimated_diameter']['kilometers']['estimated_diameter_min'],
                'est_diameter_max_km': i['estimated_diameter']['kilometers']['estimated_diameter_max'],
                'is_potentially_hazardous': i['is_potentially_hazardous_asteroid'],
                'close_approach_date': datetime.strptime(approach_data['close_approach_date'], "%Y-%m-%d").date(),
                'relative_velocity(km/h)': float(approach_data['relative_velocity']['kilometers_per_hour']),
                'astronomical_unit': float(approach_data['miss_distance']['astronomical']),
                'lunar_distance': float(approach_data['miss_distance']['lunar']),
                'miss_distance_km': float(approach_data['miss_distance']['kilometers']),
                'orbiting_body': approach_data['orbiting_body']
            }

            all_asteroids.append(asteroid_data)

            if len(all_asteroids) >= row_limit:
                break
        if len(all_asteroids) >= row_limit:
            break

    # Go to the next page of results
    start_url = data['links'].get('next')


In [28]:
!pip install pymysql

In [30]:
import mysql.connector as db

import pandas as pd

connection = db.connect(
    host = 'localhost',
    user = 'root',
    password = 'Anbu_0820',
    database = 'ds'
)
cur = connection.cursor()

In [116]:
cur.execute("""create table asteroids(
    id INT,
    name VARCHAR(100),
    absolute_magnitude_h FLOAT,
    estimated_diameter_min_km FLOAT,
    estimated_diameter_max_km FLOAT,
    is_potentially_hazardous_asteroid BOOL
)""")


In [128]:
insert_query = "INSERT INTO asteroids VALUES(%s, %s, %s, %s, %s, %s)"

for i in all_asteroids:
    id = int(i['id'])
    name = i['name']
    magnitude = float(i['absolute_magnitude_h'])
    dia_min = float(i['est_diameter_min_km'])
    dia_max = float(i['est_diameter_max_km'])
    hazard = i['is_potentially_hazardous']
    values = (id, name, magnitude, dia_min, dia_max, hazard)
    cur.execute(insert_query,values)


In [130]:
connection.commit()

In [3]:
cur.execute("""create table close_approach(
    neo_reference_id INT,
    close_approach_date DATE,
    relative_velocity_kmph FLOAT,
    AU FLOAT,
    miss_distance_km FLOAT,
    miss_distance_lunar FLOAT,
    orbiting_body TEXT 
)""")


In [38]:
insert_approach = "INSERT INTO close_approach VALUES(%s, %s, %s, %s, %s, %s, %s)"

for i in all_asteroids:
    id_neo = int(i['id'])
    close_appr = i['close_approach_date']
    speed = float(i['relative_velocity(km/h)'])
    AU = float(i['astronomical_unit'])
    miss_dist = float(i['miss_distance_km'])
    lunar = float(i['lunar_distance'])
    orbiting_body = i['orbiting_body']
    values = (id_neo, close_appr, speed, AU, miss_dist, lunar,orbiting_body)
    cur.execute(insert_approach,values)
    

In [40]:
connection.commit()

In [23]:
!echo $PATH

$PATH


In [47]:
cursor.commit()

NameError: name 'cursor' is not defined